# BIPO BehaviorBench Evaluation

This notebook evaluates the precalculated BIPO steering vectors on the adapted BehaviorBench development and held-out test sets. Calibration uses development items only; test responses are appended to the shared steering results table.

In [ ]:
from __future__ import annotations

import csv
import hashlib
import json
import random
import sys
from pathlib import Path
from typing import Any

import numpy as np
import torch

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / 'src'))
sys.path.insert(0, str(ROOT / 'scripts'))
sys.path.insert(0, str(ROOT / 'data' / 'behaviorbench' / 'adapted'))

from prepare_data import score_response
from run_caa import _layer_modules, _load_model, build_vector
from steering.common_config import MODEL_NAME, SEED, score_to_coeff
from steering.data import load_trait_data
from steering.formatting import format_pair
from steering.hooks import add_steering_hook

TRAITS = ('openness', 'conscientiousness', 'extraversion', 'agreeableness', 'neuroticism')
TARGETS = (1, 2, 3, 4, 5)
CALIBRATION_TARGETS = (1, 3, 5)
LAYER = 15
K_GRID = (0.25, 0.5, 1.0, 2.0, 4.0)
K_SWEEP_GRID = (0.25, 0.5, 1.0, 2.0, 4.0, 8.0)
PAIR_COEFFICIENTS = (-4.0, -2.0, 0.0, 2.0, 4.0)
BATCH_SIZE = 4
RESULT_FIELDS = ('method', 'trait', 'layer', 'k', 'coeff', 'target_score', 'context', 'metric', 'value', 'ci_low', 'ci_high', 'n', 'seed')

In [2]:
def load_benchmark_items(development_path: Path, test_path: Path) -> tuple[list[dict[str, Any]], list[dict[str, Any]]]:
    development = json.loads(development_path.read_text(encoding='utf-8'))
    test = json.loads(test_path.read_text(encoding='utf-8'))
    return development, test


def benchmark_prompt(item: dict[str, Any]) -> str:
    choices = 'Yes or No' if item['response_type'] == 'yes_no' else 'A or B'
    return (
        'Answer the user question using exactly one requested choice. '
        f'Reply with only {choices}, with no explanation.\n\n{item["scenario"]}'
    )


def stable_item_seed(item_id: str, target: int, seed: int) -> int:
    digest = hashlib.sha256(f'{seed}:{target}:{item_id}'.encode()).digest()
    return int.from_bytes(digest[:8], 'big')


def counterbalance_item(item: dict[str, Any], seed: int, target: int) -> dict[str, Any]:
    if item['response_type'] != 'forced_choice':
        return item
    if random.Random(stable_item_seed(item['id'], target, seed)).randrange(2) == 0:
        return item
    displayed = dict(item)
    displayed['options'] = [dict(option, key=key) for key, option in zip(('A', 'B'), reversed(item['options']))]
    return displayed


def load_bipo_vectors(vector_dir: Path, traits: tuple[str, ...] = TRAITS, vector_suffix: str = '') -> dict[str, torch.Tensor]:
    vectors = {}
    for trait in traits:
        vector = torch.load(vector_dir / f'{trait}{vector_suffix}.pt', map_location='cpu', weights_only=True).float()
        if vector.ndim != 1:
            raise ValueError(f'{trait}: expected a one-dimensional vector, got {tuple(vector.shape)}')
        vectors[trait] = vector
    return vectors

In [ ]:
def load_evaluation_model() -> tuple[Any, Any, str]:
    try:
        model, tokenizer = _load_model()
        runtime = '4-bit NF4'
    except ImportError as error:
        if 'bitsandbytes' not in str(error):
            raise
        from transformers import AutoModelForCausalLM, AutoTokenizer
        device = 'mps' if torch.backends.mps.is_available() else 'cpu'
        dtype = torch.float16 if device == 'mps' else torch.float32
        tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, local_files_only=True)
        model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, local_files_only=True, torch_dtype=dtype, low_cpu_mem_usage=True).to(device)
        runtime = f'unquantized {dtype} fallback'
    tokenizer.padding_side = 'left'
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    model.eval()
    return model, tokenizer, runtime


def generate_responses(model: Any, tokenizer: Any, items: list[dict[str, Any]], vector: torch.Tensor, coefficient: float, layer: int = LAYER) -> list[str]:
    device = next(model.parameters()).device
    handle = add_steering_hook(_layer_modules(model)[layer], coefficient, vector)
    responses = []
    try:
        for start in range(0, len(items), BATCH_SIZE):
            batch = items[start:start + BATCH_SIZE]
            encoded = tokenizer.apply_chat_template(
                [[{'role': 'user', 'content': benchmark_prompt(item)}] for item in batch],
                tokenize=True, add_generation_prompt=True, return_tensors='pt', padding=True,
            )
            input_ids = encoded.input_ids.to(device)
            attention_mask = encoded.attention_mask.to(device)
            with torch.inference_mode():
                output = model.generate(input_ids, attention_mask=attention_mask, do_sample=False, max_new_tokens=8, use_cache=True, pad_token_id=tokenizer.eos_token_id)
            responses.extend(tokenizer.batch_decode(output[:, input_ids.shape[-1]:], skip_special_tokens=True))
    finally:
        handle.remove()
    return [response.strip() for response in responses]


def score_responses(items: list[dict[str, Any]], responses: list[str]) -> tuple[list[float], float]:
    scores = [score_response(item, response) for item, response in zip(items, responses)]
    valid = [float(score) for score in scores if score is not None]
    invalid_rate = 1.0 - len(valid) / len(scores) if scores else float('nan')
    return valid, invalid_rate


def bootstrap(values: list[float], seed: int) -> tuple[float, float]:
    if not values:
        return float('nan'), float('nan')
    rng = np.random.default_rng(seed)
    means = rng.choice(values, (1000, len(values)), replace=True).mean(axis=1)
    return float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5))

In [6]:
def append_result_rows(path: Path, rows: list[dict[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    exists = path.exists() and path.stat().st_size > 0
    with path.open('a', newline='', encoding='utf-8') as handle:
        writer = csv.DictWriter(handle, fieldnames=RESULT_FIELDS)
        if not exists:
            writer.writeheader()
        writer.writerows(rows)


def evaluate_bipo_behaviorbench(
    vector_dir: Path, development_path: Path, test_path: Path, results_path: Path, raw_path: Path, seed: int = SEED,
) -> dict[str, Any]:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    development, test = load_benchmark_items(development_path, test_path)
    vectors = load_bipo_vectors(vector_dir)
    model, tokenizer, runtime = load_evaluation_model()
    hidden_size = int(model.config.hidden_size)
    if any(vector.numel() != hidden_size for vector in vectors.values()):
        raise ValueError(f'BIPO vector width does not match model hidden size {hidden_size}')
    result_rows = []
    raw_path.parent.mkdir(parents=True, exist_ok=True)
    with raw_path.open('w', newline='', encoding='utf-8') as handle:
        raw_writer = csv.DictWriter(handle, fieldnames=('item_id', 'trait', 'target_score', 'response', 'score', 'valid', 'seed'))
        raw_writer.writeheader()
        for trait in TRAITS:
            vector = vectors[trait]
            dev_items = [item for item in development if item['trait'].lower() == trait]
            test_items = [item for item in test if item['trait'].lower() == trait]
            calibration_items = dev_items[:min(20, len(dev_items))]
            calibration = []
            for multiplier in K_GRID:
                responses = generate_responses(model, tokenizer, calibration_items, vector, multiplier)
                _, invalid = score_responses(calibration_items, responses)
                calibration.append((multiplier, invalid))
            passing = [multiplier for multiplier, invalid in calibration if invalid <= 0.05]
            k = max(passing) if passing else min(calibration, key=lambda pair: pair[1])[0]
            for target in TARGETS:
                displayed = [counterbalance_item(item, seed, target) for item in test_items]
                responses = generate_responses(model, tokenizer, displayed, vector, score_to_coeff(target, k))
                scores, invalid = score_responses(displayed, responses)
                low, high = bootstrap(scores, seed)
                result_rows.append({'method': 'bipo', 'trait': trait, 'layer': LAYER, 'k': k, 'coeff': score_to_coeff(target, k), 'target_score': target, 'context': 'test', 'metric': 'behaviorbench_endorsement', 'value': float(np.mean(scores)) if scores else float('nan'), 'ci_low': low, 'ci_high': high, 'n': len(scores), 'seed': seed})
                result_rows.append({'method': 'bipo', 'trait': trait, 'layer': LAYER, 'k': k, 'coeff': score_to_coeff(target, k), 'target_score': target, 'context': 'test', 'metric': 'behaviorbench_invalid_rate', 'value': invalid, 'ci_low': '', 'ci_high': '', 'n': len(test_items), 'seed': seed})
                for item, response in zip(displayed, responses):
                    score = score_response(item, response)
                    raw_writer.writerow({'item_id': item['id'], 'trait': trait, 'target_score': target, 'response': response, 'score': score, 'valid': score is not None, 'seed': seed})
            means = []
            for target in TARGETS:
                rows = [row['value'] for row in result_rows if row['method'] == 'bipo' and row['trait'] == trait and row['metric'] == 'behaviorbench_endorsement' and row['target_score'] == target]
                means.append(rows[-1])
            result_rows.append({'method': 'bipo', 'trait': trait, 'layer': LAYER, 'k': k, 'coeff': 0, 'target_score': 3, 'context': 'test', 'metric': 'dose_slope', 'value': float(np.polyfit(TARGETS, means, 1)[0]), 'ci_low': '', 'ci_high': '', 'n': len(test_items), 'seed': seed})
    append_result_rows(results_path, result_rows)
    config_path = results_path.parent / 'bipo' / 'config.json'
    config_path.parent.mkdir(parents=True, exist_ok=True)
    config_path.write_text(json.dumps({'method': 'bipo', 'layer': LAYER, 'seed': seed, 'runtime': runtime}, indent=2) + '\n', encoding='utf-8')
    return {'rows': len(result_rows), 'runtime': runtime}

In [18]:
def response_logprob(model: Any, formatted: Any, device: torch.device) -> float:
    input_ids = formatted.input_ids.to(device)
    attention_mask = formatted.attention_mask.to(device)
    with torch.inference_mode():
        logits = model(input_ids=input_ids, attention_mask=attention_mask, use_cache=False).logits
    log_probs = logits[:, :-1].log_softmax(dim=-1)
    token_ids = input_ids[:, 1:]
    response_mask = formatted.response_mask[:, 1:].to(device)
    values = log_probs.gather(-1, token_ids.unsqueeze(-1)).squeeze(-1)[response_mask]
    return float(values.mean().item())


def pair_margin(model: Any, tokenizer: Any, row: dict[str, Any], vector: torch.Tensor, coefficient: float) -> float:
    device = next(model.parameters()).device
    positive = format_pair(tokenizer, row['prompt'], row['pos'])
    negative = format_pair(tokenizer, row['prompt'], row['neg'])
    handle = add_steering_hook(_layer_modules(model)[LAYER], coefficient, vector)
    try:
        return response_logprob(model, positive, device) - response_logprob(model, negative, device)
    finally:
        handle.remove()


def diagnose_bipo_pairs(vector_dir: Path, output_path: Path, coefficients: tuple[float, ...] = PAIR_COEFFICIENTS, seed: int = SEED) -> dict[str, Any]:
    vectors = load_bipo_vectors(vector_dir)
    model, tokenizer, runtime = load_evaluation_model()
    hidden_size = int(model.config.hidden_size)
    if any(vector.numel() != hidden_size for vector in vectors.values()):
        raise ValueError(f'BIPO vector width does not match model hidden size {hidden_size}')
    output_path.parent.mkdir(parents=True, exist_ok=True)
    rows = []
    for trait in TRAITS:
        data = load_trait_data(trait, ROOT / 'data' / 'pairs', seed=42)
        vector = vectors[trait]
        for split, examples in (('train', data.train), ('dev', data.dev)):
            for coefficient in coefficients:
                margins = [pair_margin(model, tokenizer, example, vector, coefficient) for example in examples]
                low, high = bootstrap(margins, seed)
                rows.append({
                    'method': 'bipo', 'trait': trait, 'layer': LAYER, 'split': split,
                    'coefficient': coefficient, 'value': float(np.mean(margins)),
                    'ci_low': low, 'ci_high': high, 'n': len(margins), 'seed': seed,
                })
    fields = ('method', 'trait', 'layer', 'split', 'coefficient', 'value', 'ci_low', 'ci_high', 'n', 'seed')
    with output_path.open('w', newline='', encoding='utf-8') as handle:
        writer = csv.DictWriter(handle, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)
    return {'rows': len(rows), 'runtime': runtime, 'output': str(output_path)}

In [19]:
paths = {
    'vector_dir': ROOT / 'artifacts' / 'steering_vectors' / 'bipo',
    'development_path': ROOT / 'data' / 'behaviorbench' / 'adapted' / 'data' / 'development.json',
    'test_path': ROOT / 'data' / 'behaviorbench' / 'adapted' / 'data' / 'test.json',
    'results_path': ROOT / 'artifacts' / 'steering' / 'results.csv',
    'raw_path': ROOT / 'artifacts' / 'steering' / 'bipo' / 'behaviorbench_items.csv',
}
evaluation_summary = evaluate_bipo_behaviorbench(**paths)
evaluation_summary

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

{'rows': 55, 'runtime': 'unquantized torch.float16 fallback'}

In [20]:
pair_diagnostic_summary = diagnose_bipo_pairs(
    vector_dir=ROOT / 'artifacts' / 'steering_vectors' / 'bipo',
    output_path=ROOT / 'artifacts' / 'steering' / 'bipo' / 'pair_margin_diagnostics.csv',
)
pair_diagnostic_summary

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

openness: rows=330 train=0 dev=10 test=0 missing=320
openness: final train=256 dev=74
conscientiousness: rows=281 train=181 dev=100 test=0 missing=0
conscientiousness: final train=181 dev=100
extraversion: rows=288 train=188 dev=100 test=0 missing=0
extraversion: final train=188 dev=100
agreeableness: rows=195 train=0 dev=10 test=0 missing=185
agreeableness: final train=148 dev=47
neuroticism: rows=263 train=163 dev=100 test=0 missing=0
neuroticism: final train=163 dev=100


{'rows': 50,
 'runtime': 'unquantized torch.float16 fallback',
 'output': '/Users/arya/Documents/GT/CS7650 - NLP/Project/llm-personality-conditioning/artifacts/steering/bipo/pair_margin_diagnostics.csv'}

In [3]:
def evaluate_bipo_k_sweep(
    vector_dir: Path,
    development_path: Path,
    test_path: Path,
    results_path: Path,
    calibration_path: Path,
    seed: int = SEED,
    calibration_limit: int = 20,
    traits: tuple[str, ...] = TRAITS,
    vector_suffix: str = '',
    method: str = 'bipo_k_sweep',
) -> dict[str, Any]:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    development, test = load_benchmark_items(development_path, test_path)
    vectors = load_bipo_vectors(vector_dir, traits=traits, vector_suffix=vector_suffix)
    model, tokenizer, runtime = load_evaluation_model()
    hidden_size = int(model.config.hidden_size)
    if any(vector.numel() != hidden_size for vector in vectors.values()):
        raise ValueError(f'BIPO vector width does not match model hidden size {hidden_size}')

    calibration_rows = []
    result_rows = []
    for trait in traits:
        vector = vectors[trait]
        dev_items = [item for item in development if item['trait'].lower() == trait][:calibration_limit]
        test_items = [item for item in test if item['trait'].lower() == trait]
        candidates = []
        for k in K_SWEEP_GRID:
            means = []
            invalid_rates = []
            for target in CALIBRATION_TARGETS:
                displayed = [counterbalance_item(item, seed, target) for item in dev_items]
                responses = generate_responses(model, tokenizer, displayed, vector, score_to_coeff(target, k))
                scores, invalid = score_responses(displayed, responses)
                means.append(float(np.mean(scores)) if scores else float('nan'))
                invalid_rates.append(invalid)
            slope = float(np.polyfit(CALIBRATION_TARGETS, means, 1)[0]) if all(np.isfinite(means)) else float('-inf')
            rho = float(np.corrcoef(CALIBRATION_TARGETS, means)[0, 1]) if all(np.isfinite(means)) and len(set(means)) > 1 else float('nan')
            invalid_rate = float(max(invalid_rates))
            candidates.append((k, slope, rho, invalid_rate))
            calibration_rows.append({
                'method': method, 'trait': trait, 'layer': LAYER, 'k': k,
                'slope': slope, 'spearman': rho, 'invalid_rate': invalid_rate,
                'n': len(dev_items), 'seed': seed,
            })
        eligible = [candidate for candidate in candidates if candidate[2] >= 0.5 and candidate[3] <= 0.05 and candidate[1] > 0]
        selected_k, selected_slope, selected_rho, selected_invalid = max(
            eligible or candidates, key=lambda candidate: (candidate[1], -candidate[0])
        )
        test_means = []
        for target in TARGETS:
            displayed = [counterbalance_item(item, seed, target) for item in test_items]
            responses = generate_responses(model, tokenizer, displayed, vector, score_to_coeff(target, selected_k))
            scores, invalid = score_responses(displayed, responses)
            value = float(np.mean(scores)) if scores else float('nan')
            test_means.append(value)
            low, high = bootstrap(scores, seed)
            result_rows.append({
                'method': method, 'trait': trait, 'layer': LAYER, 'k': selected_k,
                'coeff': score_to_coeff(target, selected_k), 'target_score': target,
                'context': 'test', 'metric': 'behaviorbench_endorsement', 'value': value,
                'ci_low': low, 'ci_high': high, 'n': len(scores), 'seed': seed,
            })
            result_rows.append({
                'method': method, 'trait': trait, 'layer': LAYER, 'k': selected_k,
                'coeff': score_to_coeff(target, selected_k), 'target_score': target,
                'context': 'test', 'metric': 'behaviorbench_invalid_rate', 'value': invalid,
                'ci_low': '', 'ci_high': '', 'n': len(test_items), 'seed': seed,
            })
        result_rows.append({
            'method': method, 'trait': trait, 'layer': LAYER, 'k': selected_k,
            'coeff': 0, 'target_score': 3, 'context': 'test', 'metric': 'dose_slope',
            'value': float(np.polyfit(TARGETS, test_means, 1)[0]), 'ci_low': '', 'ci_high': '',
            'n': len(test_items), 'seed': seed,
        })
        print(f'{trait}: selected k={selected_k} dev_slope={selected_slope:.4f} dev_rho={selected_rho:.4f} dev_invalid={selected_invalid:.3f}')

    calibration_path.parent.mkdir(parents=True, exist_ok=True)
    with calibration_path.open('w', newline='', encoding='utf-8') as handle:
        fields = ('method', 'trait', 'layer', 'k', 'slope', 'spearman', 'invalid_rate', 'n', 'seed')
        writer = csv.DictWriter(handle, fieldnames=fields)
        writer.writeheader()
        writer.writerows(calibration_rows)
    append_result_rows(results_path, result_rows)
    return {'rows': len(result_rows), 'calibration_rows': len(calibration_rows), 'runtime': runtime, 'output': str(calibration_path)}

In [7]:
k_sweep_v2_summary = evaluate_bipo_k_sweep(
    vector_dir=ROOT / 'artifacts' / 'steering_vectors' / 'bipo',
    development_path=ROOT / 'data' / 'behaviorbench' / 'adapted' / 'data' / 'development.json',
    test_path=ROOT / 'data' / 'behaviorbench' / 'adapted' / 'data' / 'test.json',
    results_path=ROOT / 'artifacts' / 'steering' / 'results.csv',
    calibration_path=ROOT / 'artifacts' / 'steering' / 'bipo' / 'behaviorbench_k_calibration_v2.csv',
    traits=('openness', 'conscientiousness'),
    vector_suffix='-2',
    method='bipo_v2_k_sweep',
)
k_sweep_v2_summary

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
W1107 14:06:18.926000 4160 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


openness: selected k=2.0 dev_slope=0.0750 dev_rho=1.0000 dev_invalid=0.000
conscientiousness: selected k=4.0 dev_slope=0.0625 dev_rho=0.9934 dev_invalid=0.000


{'rows': 22,
 'calibration_rows': 12,
 'runtime': 'unquantized torch.float16 fallback',
 'output': '/Users/arya/Documents/GT/CS7650 - NLP/Project/llm-personality-conditioning/artifacts/steering/bipo/behaviorbench_k_calibration_v2.csv'}